# AlexNet: ImageNet Classification with Deep Convolutional Neural Networks (Krizhevsky et al., NeurIPS 2012)
### Triển Khai Từ Con Số 0 (From Scratch) Bằng Python & NumPy

> **Tác giả:** Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton  
> **Hội nghị:** NeurIPS / NIPS 2012 (Lake Tahoe, Nevada)  
> **Ý nghĩa lịch sử:** Đánh bại hoàn toàn các phương pháp thị giác truyền thống với cách biệt 10.9%, khai sinh cuộc cách mạng Deep Learning toàn cầu.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Tầng Tích Chập 2D (Conv2D):** Tự lập trình phép trượt cửa sổ và lan truyền ngược giải tích tính đạo hàm $dW, dB, dX$.
2. **Hàm kích hoạt ReLU:** Khắc phục triệt để hiện tượng triệt tiêu gradient (Vanishing Gradient) của Sigmoid/Tanh, tăng tốc hội tụ gấp 6 lần.
3. **Chuẩn hóa phản hồi cục bộ (LRN):** Mô phỏng ức chế bên (Lateral Inhibition) trong thị giác sinh học.
4. **Gộp cực đại chồng lấn (Overlapping MaxPool 3x3, stride 2):** Giảm thiểu tỷ lệ lỗi và chống quá khớp.
5. **Inverted Dropout ($p=0.5$):** Chống co cụm nơ-ron (co-adaptation), tương đương việc huấn luyện trung bình số mũ $2^N$ mạng nơ-ron.
6. **Academic Value Benchmark:** Đối chiếu định lượng sự khác biệt bản chất giữa Kỷ nguyên Tiền-AlexNet và AlexNet.

In [ ]:
import time
import numpy as np
print(f"[✓] Đã khởi tạo môi trường NumPy {np.__version__} thành công.")

## PHẦN 1: TẦNG TÍCH CHẬP 2D (CONV2D FORWARD & BACKWARD)

Phép tích chập 2D quét qua từng bản vá (patch) kích thước $k_H \times k_W$ trên tensor ảnh đầu vào và nhân vô hướng với ma trận trọng số bộ lọc:
$$\text{Out}[n, c_{\text{out}}, i, j] = \sum_{c_{\text{in}}} \sum_{h=0}^{k_H-1} \sum_{w=0}^{k_W-1} X_{\text{padded}}[n, c_{\text{in}}, i \cdot s + h, j \cdot s + w] \cdot W[c_{\text{out}}, c_{\text{in}}, h, w] + b[c_{\text{out}}]$$

In [ ]:
class Conv2D:
    """
    Tầng Tích chập 2D cài đặt từ con số 0.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        std = np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size).astype(np.float32) * std
        self.bias = np.zeros((out_channels, 1), dtype=np.float32)

        self.d_weights = np.zeros_like(self.weights)
        self.d_bias = np.zeros_like(self.bias)
        self.x = None

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        kH = kW = self.kernel_size
        s = self.stride
        p = self.padding

        if p > 0:
            x_padded = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant')
        else:
            x_padded = x

        out_H = (H + 2 * p - kH) // s + 1
        out_W = (W + 2 * p - kW) // s + 1
        out = np.zeros((N, self.out_channels, out_H, out_W), dtype=np.float32)

        for n in range(N):
            for c_out in range(self.out_channels):
                w = self.weights[c_out]
                b = self.bias[c_out]
                for i in range(out_H):
                    for j in range(out_W):
                        h_start, w_start = i * s, j * s
                        patch = x_padded[n, :, h_start:h_start+kH, w_start:w_start+kW]
                        out[n, c_out, i, j] = np.sum(patch * w) + b

        return out

    def backward(self, d_out):
        N, C, H, W = self.x.shape
        kH = kW = self.kernel_size
        s = self.stride
        p = self.padding
        out_H, out_W = d_out.shape[2], d_out.shape[3]

        if p > 0:
            x_padded = np.pad(self.x, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant')
            dx_padded = np.zeros_like(x_padded)
        else:
            x_padded = self.x
            dx_padded = np.zeros_like(self.x)

        self.d_weights.fill(0.0)
        self.d_bias.fill(0.0)

        for n in range(N):
            for c_out in range(self.out_channels):
                self.d_bias[c_out] += np.sum(d_out[n, c_out])
                w = self.weights[c_out]
                for i in range(out_H):
                    for j in range(out_W):
                        h_start, w_start = i * s, j * s
                        patch = x_padded[n, :, h_start:h_start+kH, w_start:w_start+kW]
                        grad = d_out[n, c_out, i, j]
                        self.d_weights[c_out] += patch * grad
                        dx_padded[n, :, h_start:h_start+kH, w_start:w_start+kW] += w * grad

        if p > 0:
            dx = dx_padded[:, :, p:-p, p:-p]
        else:
            dx = dx_padded
        return dx

print("[✓] Đã định nghĩa Conv2D với cơ chế Forward & Backward hoàn chỉnh.")

## PHẦN 2: HÀM KÍCH HOẠT PHI TUYẾN: RELU vs TANH

- **ReLU (Rectified Linear Unit):** $f(x) = \max(0, x)$. Đạo hàm bằng $1$ với mọi $x > 0$. Giúp giải phóng mạng nơ-ron sâu khỏi bẫy triệt tiêu gradient.
- **Tanh:** $f(x) = \tanh(x)$. Bão hòa ở cả hai cực âm và dương (đạo hàm tiệm cận về 0 khi $|x|$ lớn).

In [ ]:
class ReLU:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return np.maximum(0.0, x)

    def backward(self, d_out):
        return d_out * (self.x > 0).astype(np.float32)


class Tanh:
    def __init__(self):
        self.out = None

    def forward(self, x):
        self.out = np.tanh(x)
        return self.out

    def backward(self, d_out):
        return d_out * (1.0 - self.out ** 2)

print("[✓] Đã khởi tạo ReLU và Tanh.")

## PHẦN 3: LRN & OVERLAPPING MAX POOLING

1. **Local Response Normalization (LRN):**
$$b_{x,y}^i = \frac{a_{x,y}^i}{\left(k + \frac{\alpha}{n} \sum_{j=i-n/2}^{i+n/2} (a_{x,y}^j)^2\right)^\beta}$$

2. **Overlapping Max Pooling:** Cửa sổ $3 \times 3$, bước trượt $2$ ($s=3 > z=2$).

In [ ]:
class LocalResponseNormalization:
    def __init__(self, n=5, k=2.0, alpha=1e-4, beta=0.75):
        self.n = n
        self.k = k
        self.alpha = alpha
        self.beta = beta
        self.x = None
        self.scale = None

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        pad = self.n // 2
        padded_sq = np.pad(x ** 2, ((0, 0), (pad, pad), (0, 0), (0, 0)), mode='constant')
        scale = np.zeros_like(x)

        for i in range(C):
            scale[:, i, :, :] = self.k + (self.alpha / self.n) * np.sum(padded_sq[:, i:i+self.n, :, :], axis=1)

        self.scale = scale
        out = x / (scale ** self.beta)
        return out

    def backward(self, d_out):
        return d_out / (self.scale ** self.beta + 1e-9)


class OverlappingMaxPool2D:
    def __init__(self, kernel_size=3, stride=2):
        self.kernel_size = kernel_size
        self.stride = stride
        self.x = None

    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        k = self.kernel_size
        s = self.stride
        out_H = (H - k) // s + 1
        out_W = (W - k) // s + 1
        out = np.zeros((N, C, out_H, out_W), dtype=np.float32)

        for n in range(N):
            for c in range(C):
                for i in range(out_H):
                    for j in range(out_W):
                        h_s, w_s = i * s, j * s
                        window = x[n, c, h_s:h_s+k, w_s:w_s+k]
                        out[n, c, i, j] = np.max(window)
        return out

    def backward(self, d_out):
        N, C, H, W = self.x.shape
        k = self.kernel_size
        s = self.stride
        out_H, out_W = d_out.shape[2], d_out.shape[3]
        dx = np.zeros_like(self.x)

        for n in range(N):
            for c in range(C):
                for i in range(out_H):
                    for j in range(out_W):
                        h_s, w_s = i * s, j * s
                        window = self.x[n, c, h_s:h_s+k, w_s:w_s+k]
                        max_val = np.max(window)
                        mask = (window == max_val)
                        dx[n, c, h_s:h_s+k, w_s:w_s+k] += mask * d_out[n, c, i, j] / (np.sum(mask) + 1e-12)
        return dx

print("[✓] Đã sẵn sàng LRN và Overlapping MaxPool.")

## PHẦN 4: DROPOUT & SOFTMAX CROSS-ENTROPY

- **Inverted Dropout:** Chia tỷ lệ $1 / (1 - p)$ trong quá trình huấn luyện để chế độ suy luận hoàn toàn không tốn thêm chi phí tính toán.
- **Softmax Cross-Entropy:** Tự động trừ max để ổn định số học và tính đạo hàm $\frac{\partial L}{\partial z} = \frac{p - y}{N}$.

In [ ]:
class Dropout:
    def __init__(self, p=0.5):
        self.p = p
        self.mask = None
        self.mode = 'train'

    def forward(self, x):
        if self.mode == 'train' and self.p > 0:
            self.mask = (np.random.rand(*x.shape) >= self.p).astype(np.float32) / (1.0 - self.p)
            return x * self.mask
        return x

    def backward(self, d_out):
        if self.mode == 'train' and self.p > 0:
            return d_out * self.mask
        return d_out


class Dense:
    def __init__(self, in_features, out_features):
        self.in_features = in_features
        self.out_features = out_features
        std = np.sqrt(2.0 / in_features)
        self.weights = np.random.randn(in_features, out_features).astype(np.float32) * std
        self.bias = np.ones((1, out_features), dtype=np.float32) * 0.1

        self.d_weights = np.zeros_like(self.weights)
        self.d_bias = np.zeros_like(self.bias)
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.weights) + self.bias

    def backward(self, d_out):
        self.d_weights = np.dot(self.x.T, d_out)
        self.d_bias = np.sum(d_out, axis=0, keepdims=True)
        return np.dot(d_out, self.weights.T)


class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.targets = None

    def forward(self, logits, targets):
        self.targets = targets
        N = logits.shape[0]
        shifted = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        self.probs = exp_scores / (np.sum(exp_scores, axis=1, keepdims=True) + 1e-12)
        correct_logprobs = -np.log(self.probs[np.arange(N), targets] + 1e-12)
        return np.mean(correct_logprobs)

    def backward(self):
        N = self.probs.shape[0]
        d_logits = self.probs.copy()
        d_logits[np.arange(N), self.targets] -= 1.0
        d_logits /= N
        return d_logits

print("[✓] Đã định nghĩa Dropout, Dense và SoftmaxCrossEntropy.")

## PHẦN 5: LẮP RÁP MÔ HÌNH MINI-ALEXNET

Lắp ráp toàn bộ các tầng: Conv1 $\rightarrow$ ReLU $\rightarrow$ LRN $\rightarrow$ MaxPool $\rightarrow$ Conv2 $\rightarrow$ ReLU $\rightarrow$ MaxPool $\rightarrow$ FC1 $\rightarrow$ ReLU $\rightarrow$ Dropout $\rightarrow$ FC2.

In [ ]:
from alexnet_scratch import MiniAlexNet

model = MiniAlexNet(in_channels=3, num_classes=4, use_relu=True, use_dropout=True, use_lrn=True)
print(f"[*] Khởi tạo thành công MiniAlexNet.")
print(f"[*] Số lượng tầng có trọng số: {len(model.trainable_layers)} tầng (Conv1, Conv2, FC1, FC2).")

## PHẦN 6: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

So sánh hiệu năng thực nghiệm giữa:
- **Mô hình Tiền-AlexNet:** Dùng Tanh, không có Dropout hay LRN $\rightarrow$ Bị triệt tiêu gradient, gradient norm tầng dưới suy hao, hội tụ chậm.
- **Mô hình Đột Phá AlexNet:** ReLU + LRN + Dropout + SGD Momentum $\rightarrow$ Gradient lan truyền mạnh mẽ, hội tụ siêu tốc, đạt độ chính xác cao.

In [ ]:
from alexnet_scratch import run_academic_value_comparison

# Chạy thử nghiệm đối chiếu trực tiếp
run_academic_value_comparison()

## 🎓 TỔNG KẾT & DI SẢN CỦA ALEXNET (NEURIPS 2012)

1. **Chấm dứt kỷ nguyên Feature Engineering:** Chứng minh rằng mô hình học sâu có thể học các biểu diễn thị giác phân cấp trực tiếp từ điểm ảnh thô tốt hơn mọi thuật toán thủ công (SIFT, HOG).
2. **ReLU là tiêu chuẩn vàng:** Trở thành hàm kích hoạt mặc định cho hầu hết mọi kiến trúc mạng nơ-ron tích chập và thị giác máy tính sau này.
3. **Dropout:** Kỹ thuật điều hòa kinh điển giúp huấn luyện các mạng nơ-ron khổng lồ mà không sợ quá khớp.
4. **Kỷ nguyên GPU Computing:** Mở ra trào lưu đưa GPU vào huấn luyện học sâu song song, dẫn lối đến các siêu mô hình ngày nay (LLMs, Diffusion Models).